In [3]:
import yfinance as yf
import numpy as np
import pandas as pd
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.env_checker import check_env
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor


In [4]:
tickers = ["AAPL", "MSFT", "NVDA", "GOOGL", "AMZN"]

data = yf.download(
    tickers,
    start="2018-01-01",
    end="2023-01-01",
    interval="1d"
)


# Séparer prix et volume
price_df = data["Close"]
volume_df = data["Volume"]

print(price_df.head())
print(volume_df.head())

[*********************100%***********************]  5 of 5 completed

Ticker           AAPL       AMZN      GOOGL       MSFT      NVDA
Date                                                            
2018-01-02  40.304188  59.450500  53.220631  78.870369  4.928266
2018-01-03  40.297153  60.209999  54.128628  79.237427  5.252614
2018-01-04  40.484337  60.479500  54.338890  79.934822  5.280302
2018-01-05  40.945267  61.457001  55.059433  80.925850  5.325049
2018-01-08  40.793190  62.343498  55.253830  81.008446  5.488213
Ticker           AAPL      AMZN     GOOGL      MSFT       NVDA
Date                                                          
2018-01-02  102223600  53890000  31766000  22483800  355616000
2018-01-03  118071600  62176000  31318000  26061400  914704000
2018-01-04   89738400  60442000  26052000  21912000  583268000
2018-01-05   94640000  70894000  30250000  23407100  580124000
2018-01-08   82271200  85590000  24644000  22113000  881216000


In [ ]:
def prepare_features(price_df: pd.DataFrame, volume_df: pd.DataFrame):
    """
    price_df  : DataFrame index=date, columns=tickers, values=close
    volume_df : DataFrame index=date, columns=tickers, values=volume

    Retourne un DataFrame multi-index colonnes:
    (ticker, feature)
    """
    tickers = list(price_df.columns)

    feats = {}
    for t in tickers:
        px = price_df[t].copy()
        vol = volume_df[t].copy()

        df_t = pd.DataFrame(index=price_df.index)
        df_t["close"] = px
        df_t["ret_1"] = px.pct_change(1)
        df_t["ret_5"] = px.pct_change(5)
        df_t["vol_20"] = px.pct_change().rolling(20).std() * np.sqrt(252)

        vol_mean = vol.rolling(20).mean()
        vol_std = vol.rolling(20).std()
        df_t["vol_z"] = (vol - vol_mean) / (vol_std + 1e-8)

        feats[t] = df_t

    panel = pd.concat(feats, axis=1)
    panel = panel.dropna().copy()
    return panel